# py-sommer Implementation Starter Notebook

This notebook provides a reproducible starting point for mixed-model implementation workflows using `pysommer`.

## 1. Environment and Kernel Verification

Run executable checks for Python version, active kernel, and package availability (`numpy`, `pysommer`).

In [3]:
import importlib
import platform
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

np_spec = importlib.util.find_spec("numpy")
ps_spec = importlib.util.find_spec("pysommer")
print("numpy available:", np_spec is not None)
print("pysommer available:", ps_spec is not None)

import numpy as np
import pysommer

print("numpy version:", np.__version__)
print("pysommer module:", pysommer.__name__)

Python: 3.13.1
Platform: macOS-26.2-arm64-arm-64bit-Mach-O
numpy available: True
pysommer available: True
numpy version: 2.4.3
pysommer module: pysommer


## 2. Project Imports and Path Setup

Import core APIs and set a deterministic random seed.

In [2]:
from pathlib import Path

from pysommer import ism, mmes, vsm

SEED = 20260324
rng = np.random.default_rng(SEED)
print("Seed:", SEED)
print("Working directory:", Path.cwd())

Seed: 20260324
Working directory: /Users/nico/Desktop/Projects/py-sommer/notebooks


## 3. Minimal Synthetic Data for Mixed Model

Create grouped observations with an intercept, random group effect, and residual noise.

In [4]:
n_groups = 12
reps = 4
group = np.repeat(np.arange(n_groups), reps)
n = group.size

u_true = rng.normal(0.0, np.sqrt(0.8), size=(n_groups, 1))
e = rng.normal(0.0, np.sqrt(0.3), size=(n, 1))
y = 2.0 + u_true[group] + e

print("n observations:", n)
print("n groups:", n_groups)
print("y shape:", y.shape)

n observations: 48
n groups: 12
y shape: (48, 1)


## 4. Design Matrices Construction (`X`, `Z`, `K`)

Build fixed-effect matrix `X`, random-effect incidence matrix `Z`, and covariance matrix `K`.

In [7]:
X = np.ones((n, 1), dtype=float)
Z = np.eye(n_groups, dtype=float)[group]
K = np.eye(n_groups, dtype=float)

print("X shape:", X.shape)
print("Z shape:", Z.shape)
print("K shape:", K.shape)
assert Z.shape[0] == X.shape[0] == y.shape[0]
assert Z.shape[1] == K.shape[0] == K.shape[1]

X shape: (48, 1)
Z shape: (48, 12)
K shape: (12, 12)


## 5. First `mmes` Fit (Matrix API)

Fit a baseline model with a limited iteration count for fast startup checks.

In [10]:
fit_base = mmes(
    Y=y,
    X=X,
    Z=[Z],
    K=[K],
    iters=35,
    method="newton_di_sp",
)

print("Converged:", fit_base["converged"])
print("Iterations:", fit_base["iterations"])

Converged: True
Iterations: 8


## 6. Inspect Core Outputs (`beta`, `theta`, `u`, convergence)

Extract and check core result structures.

In [12]:
beta = np.asarray(fit_base["beta"])
theta = np.asarray(fit_base["theta"])
u = fit_base["u"]

print("beta:", beta.ravel())
print("theta:", theta.ravel())
print("u terms:", len(u))
print("u[0] shape:", np.asarray(u[0]).shape)
print("converged:", fit_base["converged"])

beta: [1.73207233]
theta: [0.78095002 0.30564563]
u terms: 1
u[0] shape: (12, 1)
converged: True


## 7. Basic Prediction and Residual Checks

Compute fitted values and residual diagnostics.

In [13]:
u_hat = np.asarray(fit_base["u"][0])
y_hat = X @ beta + Z @ u_hat
resid = y - y_hat

rmse = float(np.sqrt(np.mean(resid**2)))
resid_mean = float(np.mean(resid))

print("RMSE:", round(rmse, 6))
print("Residual mean:", round(resid_mean, 6))

RMSE: 0.485259
Residual mean: 0.0


## 8. Second Fit with `ai_mme_sp` for Solver Comparison

Fit the same model with the alternative solver and compare variance components.

In [16]:
fit_ai = mmes(
    Y=y,
    X=X,
    Z=[Z],
    K=[K],
    iters=35,
    method="ai_mme_sp",
)

theta_base = np.asarray(fit_base["theta"]).reshape(-1)
theta_ai = np.asarray(fit_ai["theta"]).reshape(-1)
diff = np.abs(theta_base - theta_ai)

print("theta (newton):", theta_base)
print("theta (ai):", theta_ai)
print("|difference|:", diff)

theta (newton): [0.78095002 0.30564563]
theta (ai): [0.78095002 0.30564563]
|difference|: [3.13415960e-13 3.99680289e-15]


## 9. Quick Formula-Style Fit with `vsm`/`ism`

Fit an equivalent random-intercept model using formula-style syntax.

In [17]:
data = {
    "y": y.ravel(),
    "group": group,
}

fit_formula = mmes(
    fixed="y ~ 1",
    random=[vsm(ism("group"))],
    data=data,
    iters=35,
)

print("formula beta:", np.asarray(fit_formula["beta"]).ravel())
print("formula theta:", np.asarray(fit_formula["theta"]).ravel())
print("random names:", fit_formula.get("random_names"))

formula beta: [1.73207233]
formula theta: [0.78095002 0.30564563]
random names: ['ism(group)']


## 10. Sanity Assertions for Reproducible Start State

Lock in finite outputs and expected dimensions as a stable baseline for future implementation work.

In [18]:
assert np.isfinite(theta_base).all()
assert np.isfinite(theta_ai).all()
assert beta.shape == (1, 1)
assert np.asarray(fit_base["u"][0]).shape == (n_groups, 1)
assert y_hat.shape == y.shape
assert np.isfinite(rmse)

# Loose tolerance: different optimizers should be in the same neighborhood.
assert np.max(diff) < 0.25

print("All starter assertions passed.")

All starter assertions passed.
